In [ ]:
import torch
import numpy as np
import random
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import collections
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
dataset = load_dataset("SetFit/amazon_reviews_multi_es")

In [ ]:
model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizador = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenizar(ejemplo):
    return tokenizador(
        ejemplo["text"],
        truncation=True,
        padding=False,
        max_length=256
    )

datos_tokenizados = dataset.map(tokenizar, batched=True)
datos_tokenizados = datos_tokenizados.remove_columns(["text", "label_text"])
datos_tokenizados.set_format("torch")

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizador)
modelo = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value", "key"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

modelo = get_peft_model(modelo, lora_config)

modelo.print_trainable_parameters()

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="modelo-ajustado",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    optim="adamw_torch",

    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    logging_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=modelo,
    args=training_args,
    train_dataset=datos_tokenizados["train"].shuffle(seed=42).select(range(20000)),
    eval_dataset=datos_tokenizados["validation"].shuffle(seed=42).select(range(4000)),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

In [ ]:
trainer.train()

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)

train_loss = logs[logs["loss"].notna()].copy()
train_loss["epoch_int"] = train_loss["epoch"].astype(int)
train_loss_epoch = train_loss.groupby("epoch_int")["loss"].mean()

eval_loss = logs[logs["eval_loss"].notna()].copy()
eval_loss["epoch_int"] = eval_loss["epoch"].astype(int)
eval_loss_epoch = eval_loss.groupby("epoch_int")["eval_loss"].mean()

best_epoch = eval_loss_epoch.idxmin()
best_value = eval_loss_epoch.min()

plt.figure(figsize=(8,5))

plt.plot(train_loss_epoch.index, train_loss_epoch.values, marker="o", label="Train Loss")
plt.plot(eval_loss_epoch.index, eval_loss_epoch.values, marker="o", label="Validation Loss")

plt.scatter(best_epoch, best_value)
plt.text(best_epoch, best_value, "  Mejor modelo", fontsize=10)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss (por epoch REAL)")
plt.legend()
plt.grid(alpha=0.3)

plt.show()